In [6]:
import cv2
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import numpy as np
import pyttsx3 # for audio using pytext3

model_name = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

engine = pyttsx3.init() #audio engine 

def generate_caption(frames):
    """Generate a collective caption for a list of frames."""
    # Average the frames to get a single representative image
    avg_frame = np.mean(frames, axis=0).astype(np.uint8)
    image = Image.fromarray(cv2.cvtColor(avg_frame, cv2.COLOR_BGR2RGB))
    
    # Generate the caption
    inputs = processor(image, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    return processor.decode(outputs[0], skip_special_tokens=True)

def read_caption_aloud(caption): 
    """Read the caption aloud using text-to-speech."""
    engine.say(caption)
    engine.runAndWait()

def main():
    """Capture video from webcam and process frames for batch captioning."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print("Press 'q' to exit.")

    frame_buffer = []  # Store a batch of frames
    batch_size = 30    # Process 30 frames at a time (about 1 second at 30 fps)
    last_caption = None #for last generated caption 
    

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break

        # Add frame to the buffer
        frame_buffer.append(frame)

        # Process the buffer when enough frames are collected
        if len(frame_buffer) == batch_size:
            caption = generate_caption(frame_buffer)
            print("Caption:", caption)
            if caption != last_caption:
                read_caption_aloud(caption)
                last_caption = caption
            frame_buffer = []  # Clear the buffer

        # Display the current frame
        cv2.imshow("Real-Time Image Captioning", frame)

        # Exit on pressing 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Press 'q' to exit.
Caption: a woman in a red shirt and sunglasses
Caption: a blur of a person walking in a room
Caption: a blur of a person in a room
Caption: a black background with a white border
Caption: a black background with a white border
Caption: a black background with a white border
Caption: a black background with a white border
Caption: a black background with a white border
Caption: a woman with blue hair
